In [1]:
####################################
#ENVIRONMENT SETUP

In [2]:
#LIBRARIES
import os, sys

import numpy as np
import math

import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import xarray as xr

import pickle 

from tqdm import tqdm

In [3]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
from CLASSES_Directories import DirectoryManager_Class

In [4]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "InitialFigures")
dataType = "SurfaceVariableAnimations_Structured"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)

 DirectoryManager_Class Summary
 Main Directory:           /glade/u/home/aroseman/Projects/Regional-MPAS-Project
 Main Scratch Directory:   /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project
 Scratch Directory:        /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0
 Main Output Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT
 Main Code Directory:      /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles
 Current Code Directory:   /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/DataAnalysis/MPAS_Model_Data/InitialFigures



In [5]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class,DataOperator_Class

In [6]:
RunType = ("TRACER","MOIST","NSSL")
SimulationTime = ("2022-06-30","2022-07-03")
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

Found 264/289 files matching expected times.
Opened history file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_NSSL/backup_RESTART2/history_cartesian/history.2022-06-30_00.00.00.latlon.nc
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_NSSL/backup_RESTART2/diag_cartesian/diag.2022-06-30_00.00.00.latlon.nc

=== MPAS Structured (lat-lon) Model Data Summary ===
 Region:         TRACER
 Case:           MOIST
 Microphysics:   NSSL
 Resolution:     20-1km
 Time Step:      15mins
 Time Range:     2022-06-30 to 2022-07-03
 Coordinates:    ['latitude', 'longitude', 'nVertLevels', 'nVertLevelsP1', 'nSoilLevels']
 # History Files:264
 # Diag Files:   264
 # Time Steps:   264
 Data Directory: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_NSSL
 Static File:    TRACER_regional5250_scaled3_x20.8355

In [7]:
###############
#FUNCTIONS

In [ ]:
def InitiateMatrix(variableSubset):
    #3d variable case
    if ("nVertLevels" in variableSubset.dims):
        output = np.zeros((ModelData.Ntime,ModelData.Nzc))
    #3d variable case
    elif ("nVertLevelsP1" in variableSubset.dims):
        output = np.zeros((ModelData.Ntime,ModelData.Nzf))
    else:  #2d variable case
        output = np.zeros((ModelData.Ntime,1))
    return output

def GetMean(variableSubset):
    variableMean = variableSubset.mean(dim=("latitude","longitude")).data
    return variableMean

In [ ]:
def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    If varName contains a '+', returns the sum of the two variables.
    """
    if '+' in varName:
        var1, var2 = varName.split('+')
        var1 = var1.strip()
        var2 = var2.strip()

        subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var1)
        subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var2)
        variableSubset = subset1 + subset2
    else:
        variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                             dataSubset_diag, dataSubset_static, varName)

    return variableSubset

def RunCalculations(varNames):
    outputDictionary={}
    
    num_times = ModelData.Ntime
    for count, t in enumerate(tqdm(range(num_times), desc="Processing timesteps")):
        # if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")
            
        #Loading Data
        [dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)

        for varName in varNames:
            if count == 0: print(f"Running for {varName}","\n")
            #Subsetting Data

            variableSubset= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)
    
            #Initializing Output
            if count == 0:
                output = InitiateMatrix(variableSubset)
                outputDictionary[varName] = output

            #Taking Mean
            variableMean = GetMean(variableSubset)
            outputDictionary[varName][t] = variableMean


    outputDictionary[varName] = output
    return outputDictionary
varNames = [
    # "u10", "v10", "q2",
    # "hfx", "qfx", "lh",
    # "rainnc+rainc", #*#*
    # "refl10cm_1km"
]
# varNames += ["w", "theta", "qv", "qc+qi", "qr", "refl10cm"]
outputDictionary = RunCalculations(varNames)

In [28]:
t=0
[dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)
DataOperator_Class.GetData_Variable(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)

# variableSubset

In [15]:
# varName='u10'
# plt.plot(outputDictionary[varName])
# varName='v10'
# plt.plot(outputDictionary[varName])
# varName = "rainnc+rainc"
# plt.plot(outputDictionary[varName])

In [43]:
# varName = "w"
# plt.contourf(outputDictionary[varName].T) 
# plt.colorbar() #nice!

In [ ]:
# Notes:
# (1) may need to subset land/water later